# Create Sample Incoming Batch

This notebook creates a small sample batch from locally available NCBI phenotype and genotype tables so the AMR system can simulate newly incoming data. It can also optionally download a few genome FASTA files for the sampled `BioSample` IDs so `/predict-fasta` can be tested end-to-end.

## 1. Configuration

Update the source paths below if your local NCBI files are stored outside the default `data/` folder. The FASTA download section is optional and only needs internet access when enabled.

In [ ]:
from __future__ import annotations

import gzip
import json
import shutil
import time
from pathlib import Path
from urllib import error, parse, request

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent

# Update these paths if your local files are stored elsewhere.
PHENOTYPE_PATH = PROJECT_ROOT / 'data' / 'phenotype.tsv'
GENOTYPE_PATH = PROJECT_ROOT / 'data' / 'genotype.tsv'

BATCH_NAME = 'batch_001'
N_BIOSAMPLES = 50
RANDOM_SEED = 42

FASTA_DOWNLOAD_ENABLED = True
FASTA_DOWNLOAD_LIMIT = 3
NCBI_TOOL = 'amr-prediction-notebook'
NCBI_EMAIL = ''
NCBI_API_KEY = ''
NCBI_REQUEST_DELAY_SECONDS = 0.34

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'incoming' / BATCH_NAME
FASTA_OUTPUT_DIR = OUTPUT_DIR / 'fasta'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT          :', PROJECT_ROOT)
print('PHENOTYPE_PATH        :', PHENOTYPE_PATH)
print('GENOTYPE_PATH         :', GENOTYPE_PATH)
print('OUTPUT_DIR            :', OUTPUT_DIR)
print('FASTA_DOWNLOAD_ENABLED:', FASTA_DOWNLOAD_ENABLED)
print('FASTA_DOWNLOAD_LIMIT  :', FASTA_DOWNLOAD_LIMIT)

if not PHENOTYPE_PATH.exists():
    raise FileNotFoundError(f'Phenotype file not found: {PHENOTYPE_PATH}')
if not GENOTYPE_PATH.exists():
    raise FileNotFoundError(f'Genotype file not found: {GENOTYPE_PATH}')


## 2. Load Raw NCBI Tables

In [ ]:
phenotype = pd.read_csv(
    PHENOTYPE_PATH,
    sep='\t',
    dtype={
        '#BioSample': 'string',
        'Antibiotic': 'string',
        'Resistance phenotype': 'string',
        'Measurement sign': 'string',
        'MIC (mg/L)': 'string',
    },
).rename(columns={'#BioSample': 'BioSample'})

genotype = pd.read_csv(
    GENOTYPE_PATH,
    sep='\t',
    dtype={
        '#BioSample': 'string',
        'Element symbol': 'string',
        'Subtype': 'string',
        'Class': 'string',
        'Subclass': 'string',
        'Method': 'string',
        '% Coverage of reference': 'string',
        '% Identity to reference': 'string',
    },
).rename(columns={'#BioSample': 'BioSample'})

phenotype['BioSample'] = phenotype['BioSample'].str.strip()
genotype['BioSample'] = genotype['BioSample'].str.strip()

print('phenotype shape:', phenotype.shape)
print('genotype shape :', genotype.shape)
display(phenotype.head())
display(genotype.head())

## 3. Sample Shared Isolates

To simulate incoming AMR records, we sample `BioSample` IDs that appear in both tables.

In [ ]:
phenotype_isolates = set(phenotype['BioSample'].dropna().unique().tolist())
genotype_isolates = set(genotype['BioSample'].dropna().unique().tolist())
shared_isolates = sorted(phenotype_isolates & genotype_isolates)

if not shared_isolates:
    raise ValueError('No shared BioSample IDs were found between phenotype and genotype tables.')

sample_size = min(N_BIOSAMPLES, len(shared_isolates))
sampled_biosamples = (
    pd.Series(shared_isolates)
    .sample(n=sample_size, random_state=RANDOM_SEED, replace=False)
    .sort_values()
    .tolist()
)

batch_phenotype = phenotype[phenotype['BioSample'].isin(sampled_biosamples)].copy()
batch_genotype = genotype[genotype['BioSample'].isin(sampled_biosamples)].copy()

manifest = pd.DataFrame({'BioSample': sampled_biosamples})

print('shared isolates total :', len(shared_isolates))
print('sampled isolates      :', len(sampled_biosamples))
print('batch phenotype rows  :', len(batch_phenotype))
print('batch genotype rows   :', len(batch_genotype))
display(manifest.head())

## 4. Save Incoming Batch Files

In [ ]:
phenotype_out = OUTPUT_DIR / 'phenotype_incoming.tsv'
genotype_out = OUTPUT_DIR / 'genotype_incoming.tsv'
manifest_out = OUTPUT_DIR / 'biosample_manifest.csv'
summary_out = OUTPUT_DIR / 'batch_summary.json'

batch_phenotype.to_csv(phenotype_out, sep='\t', index=False)
batch_genotype.to_csv(genotype_out, sep='\t', index=False)
manifest.to_csv(manifest_out, index=False)

summary = {
    'batch_name': BATCH_NAME,
    'sample_size': int(len(sampled_biosamples)),
    'phenotype_rows': int(len(batch_phenotype)),
    'genotype_rows': int(len(batch_genotype)),
    'unique_antibiotics': sorted(batch_phenotype['Antibiotic'].dropna().astype(str).unique().tolist()),
    'source_phenotype_path': str(PHENOTYPE_PATH),
    'source_genotype_path': str(GENOTYPE_PATH),
}

summary_out.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Saved files:')
print('-', phenotype_out)
print('-', genotype_out)
print('-', manifest_out)
print('-', summary_out)

display(batch_phenotype.head())
display(batch_genotype.head())

## 5. Optional Download FASTA Files

This section tries to resolve each sampled `BioSample` to an NCBI Assembly record, then downloads the assembly genomic FASTA into `data/incoming/<batch>/fasta/`.

Notes:
- It uses NCBI E-utilities (`esearch` + `esummary`) and then downloads `*_genomic.fna.gz` from the assembly FTP path.
- Some `BioSample` IDs may not resolve to any public assembly.
- NCBI may temporarily block or throttle repeated requests; when that happens the error is captured in the manifest.

In [ ]:
def build_eutils_params(extra: dict[str, str]) -> dict[str, str]:
    params = dict(extra)
    params['tool'] = NCBI_TOOL
    if NCBI_EMAIL:
        params['email'] = NCBI_EMAIL
    if NCBI_API_KEY:
        params['api_key'] = NCBI_API_KEY
    return params


def fetch_json(url: str, params: dict[str, str]) -> dict[str, object]:
    query = parse.urlencode(build_eutils_params(params))
    full_url = f'{url}?{query}'
    req = request.Request(full_url, headers={'User-Agent': NCBI_TOOL})
    try:
        with request.urlopen(req) as response:
            body = response.read().decode('utf-8', errors='replace')
    except error.HTTPError as exc:
        detail = exc.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'HTTP {exc.code} for {full_url}: {detail[:300]}') from exc
    except error.URLError as exc:
        raise RuntimeError(f'Failed to reach NCBI for {full_url}: {exc}') from exc

    try:
        return json.loads(body)
    except json.JSONDecodeError as exc:
        raise RuntimeError(f'NCBI did not return JSON for {full_url}: {body[:300]}') from exc


def resolve_assembly_for_biosample(biosample: str) -> dict[str, str] | None:
    time.sleep(NCBI_REQUEST_DELAY_SECONDS)
    search_payload = fetch_json(
        'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi',
        {
            'db': 'assembly',
            'term': f'{biosample}[BioSample]',
            'retmode': 'json',
            'retmax': '1',
        },
    )
    ids = search_payload.get('esearchresult', {}).get('idlist', [])
    if not ids:
        return None

    time.sleep(NCBI_REQUEST_DELAY_SECONDS)
    summary_payload = fetch_json(
        'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi',
        {
            'db': 'assembly',
            'id': ids[0],
            'retmode': 'json',
        },
    )
    result = summary_payload.get('result', {})
    uid = result.get('uids', [None])[0]
    if uid is None:
        return None
    doc = result.get(uid, {})

    ftp_path = doc.get('ftppath_refseq') or doc.get('ftppath_genbank')
    assembly_accession = doc.get('assemblyaccession') or ''
    assembly_name = doc.get('assemblyname') or ''
    if not ftp_path:
        return None

    return {
        'assembly_uid': str(uid),
        'assembly_accession': str(assembly_accession),
        'assembly_name': str(assembly_name),
        'ftp_path': str(ftp_path),
    }


def build_genomic_fasta_url(ftp_path: str) -> str:
    ftp_path = ftp_path.rstrip('/')
    assembly_basename = ftp_path.rsplit('/', 1)[-1]
    https_path = ftp_path.replace('ftp://', 'https://', 1)
    return f'{https_path}/{assembly_basename}_genomic.fna.gz'


def download_genomic_fasta(fasta_url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    gz_destination = destination.with_suffix(destination.suffix + '.gz')
    req = request.Request(fasta_url, headers={'User-Agent': NCBI_TOOL})
    try:
        with request.urlopen(req) as response, gz_destination.open('wb') as out_handle:
            shutil.copyfileobj(response, out_handle)
    except error.HTTPError as exc:
        detail = exc.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'HTTP {exc.code} while downloading {fasta_url}: {detail[:300]}') from exc
    except error.URLError as exc:
        raise RuntimeError(f'Failed to download {fasta_url}: {exc}') from exc

    with gzip.open(gz_destination, 'rb') as compressed, destination.open('wb') as out_handle:
        shutil.copyfileobj(compressed, out_handle)
    gz_destination.unlink(missing_ok=True)


fasta_manifest_out = OUTPUT_DIR / 'fasta_manifest.csv'
fasta_rows: list[dict[str, str]] = []

if not FASTA_DOWNLOAD_ENABLED:
    print('FASTA download is disabled. Set FASTA_DOWNLOAD_ENABLED = True to enable it.')
else:
    FASTA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    target_biosamples = sampled_biosamples[: min(FASTA_DOWNLOAD_LIMIT, len(sampled_biosamples))]
    print('Attempting FASTA download for', len(target_biosamples), 'biosamples')

    for biosample in target_biosamples:
        row = {
            'BioSample': biosample,
            'assembly_uid': '',
            'assembly_accession': '',
            'assembly_name': '',
            'fasta_url': '',
            'fasta_path': '',
            'status': 'pending',
            'error': '',
        }
        try:
            assembly_info = resolve_assembly_for_biosample(biosample)
            if assembly_info is None:
                row['status'] = 'no_public_assembly_found'
            else:
                row.update(assembly_info)
                fasta_url = build_genomic_fasta_url(assembly_info['ftp_path'])
                fasta_path = FASTA_OUTPUT_DIR / f"{biosample}__{assembly_info['assembly_accession']}.fasta"
                row['fasta_url'] = fasta_url
                row['fasta_path'] = str(fasta_path)
                time.sleep(NCBI_REQUEST_DELAY_SECONDS)
                download_genomic_fasta(fasta_url, fasta_path)
                row['status'] = 'downloaded'
        except Exception as exc:
            row['status'] = 'failed'
            row['error'] = str(exc)
        fasta_rows.append(row)

fasta_manifest = pd.DataFrame(fasta_rows)
fasta_manifest.to_csv(fasta_manifest_out, index=False)

summary = json.loads(summary_out.read_text(encoding='utf-8')) if summary_out.exists() else {}
summary.update(
    {
        'fasta_download_enabled': bool(FASTA_DOWNLOAD_ENABLED),
        'fasta_download_limit': int(FASTA_DOWNLOAD_LIMIT),
        'fasta_output_dir': str(FASTA_OUTPUT_DIR),
        'fasta_manifest_path': str(fasta_manifest_out),
        'fasta_downloaded': int((fasta_manifest['status'] == 'downloaded').sum()) if not fasta_manifest.empty else 0,
        'fasta_attempted': int(len(fasta_manifest)),
    }
)
summary_out.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Saved FASTA manifest:', fasta_manifest_out)
if not fasta_manifest.empty:
    display(fasta_manifest)
else:
    print('No FASTA rows were generated.')


## 6. Optional Quick Summary

In [ ]:
antibiotic_summary = (
    batch_phenotype.groupby('Antibiotic', dropna=False)
    .agg(
        rows=('BioSample', 'size'),
        biosamples=('BioSample', 'nunique'),
        resistant=('Resistance phenotype', lambda s: (s.astype(str).str.lower() == 'resistant').sum()),
        susceptible=('Resistance phenotype', lambda s: (s.astype(str).str.lower() == 'susceptible').sum()),
    )
    .sort_values(['rows', 'biosamples'], ascending=[False, False])
    .reset_index()
)

display(antibiotic_summary)

if (OUTPUT_DIR / 'fasta_manifest.csv').exists():
    display(pd.read_csv(OUTPUT_DIR / 'fasta_manifest.csv').head())